<a href="https://colab.research.google.com/github/josedanielisidororeyes/Advanced_Data_Engineering/blob/main/Data_Profiling_con_PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Nombre del Alumno: José Daniel Isidoro Reyes
#Matrícula: 261552
#Matería: Ingeniería de Datos Avanzada
#Nombre de la Tarea: Data Profiling con PySpark
#Fecha: 03/05/2026

In [39]:
%%capture
!pip install ydata-profiling[pyspark]
!pip install ipywidgets

In [40]:
# Carga de librerias
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
from ydata_profiling import ProfileReport

# Carga del conjunto de datos con PySpark

In [ ]:
spark  =  SparkSession.builder.appName("NYC Taxi Profiling").getOrCreate()

!wget -O taxi.csv "https://www.dropbox.com/scl/fi/ya6wwi1ouvu7b5ng00zu3/yellow_tripdata_2016-03.csv?rlkey=49gbpo35mmh7p2codjw4kcfd3&dl=1"
df =  spark.read.csv("taxi.csv", header =  True, inferSchema =  True)

--2026-05-03 17:40:50--  https://www.dropbox.com/scl/fi/ya6wwi1ouvu7b5ng00zu3/yellow_tripdata_2016-03.csv?rlkey=49gbpo35mmh7p2codjw4kcfd3&dl=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.4.18, 2620:100:601c:18::a27d:612
Connecting to www.dropbox.com (www.dropbox.com)|162.125.4.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://ucb715ddfef2af29eb2f7c5627f3.dl.dropboxusercontent.com/cd/0/inline/C_yqtchPImZQTTnzYHyPogTkbUV63x8E9G0IBX62LnDAuRRXy37XgrBYg7-VurHQhhtSursWEN62Kbc7aL0qN9X98DlsmFb04i1y8NZeTmUqTBELMlnWV-Gg44Nw_x4Z1Ks/file?dl=1# [following]
--2026-05-03 17:40:50--  https://ucb715ddfef2af29eb2f7c5627f3.dl.dropboxusercontent.com/cd/0/inline/C_yqtchPImZQTTnzYHyPogTkbUV63x8E9G0IBX62LnDAuRRXy37XgrBYg7-VurHQhhtSursWEN62Kbc7aL0qN9X98DlsmFb04i1y8NZeTmUqTBELMlnWV-Gg44Nw_x4Z1Ks/file?dl=1
Resolving ucb715ddfef2af29eb2f7c5627f3.dl.dropboxusercontent.com (ucb715ddfef2af29eb2f7c5627f3.dl.dropboxusercontent.com)... 162.125.4.15, 2620:100:601c:15::

Se inicia y configura sesión en Spark con SparkSesion. Posteriomente, se descarga el archivo de Dropbox y se transforma a un dataframe.

# Proceso de Data Profiling

## Estructura del dataset(schema, número de filas y columnas)

In [27]:
# Esquema del conjunto de datos
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)



Posteriormente, se hace una lectura del tipo de metadato por columna a través de PrintSchema. Se observa que Spark hizo un buen trabajo al inferir el tipo de dato, sin embargo, variables como RatecodeID y payment_type probablemente necesiten ser transformadas a tipo categorico para aplicaciones de aprendizaje automático.

In [28]:
# Número de filas y columnas
num_filas = df.count()
num_cols  = len(df.columns)
print(f"{num_filas} filas X {num_cols} columnas")

12210952 filas X 19 columnas


Se observa que el conjunto de datos es grande, con más de 12 millones de registros y 19 columnas.

In [29]:
# Impresión de filas filas
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+-----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|  pickup_longitude|   pickup_latitude|RatecodeID|store_and_fwd_flag| dropoff_longitude| dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+-----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|       1| 2016-03-01 00:00:00|  2016-03-01 00:07:55|              1|          2.5|-73.97674560546875| 40.76515197753906|         1|       

A modo de ilustración se imprimen los primeros 5 registros para una mejor comporesión del conjunto de datos.

## Estadísticos Descriptivos

In [30]:
tipos_numericos  =  ["double", "integer"]
columnas_numericas = [col_name for col_name, col_type in df.dtypes if col_type in tipos_numericos]
df.select(columnas_numericas).summary().show()

+-------+------------------+-------------------+------------------+-------------------+------------------+------------------+-------------------+-------------------+------------------+-------------------+---------------------+-----------------+
|summary|     trip_distance|   pickup_longitude|   pickup_latitude|  dropoff_longitude|  dropoff_latitude|       fare_amount|              extra|            mta_tax|        tip_amount|       tolls_amount|improvement_surcharge|     total_amount|
+-------+------------------+-------------------+------------------+-------------------+------------------+------------------+-------------------+-------------------+------------------+-------------------+---------------------+-----------------+
|  count|          12210952|           12210952|          12210952|           12210952|          12210952|          12210952|           12210952|           12210952|          12210952|           12210952|             12210952|         12210952|
|   mean| 6.13176976

A continuación se hace un análisis descriptivo de las variables númericas, por ejemplo, la distancia promedio por viaje es de 6.13 millas, la tarifa total promedio por viajes es de 16.04, etc. De igual manera, se identifican valores atipicos en el conjunto de datos, por ejemplo, tarifas de 429562 dolares o viajes de 19,072,628.8 millas, que necesitan mayor investigación. Además de distancias de viaje iguales a 0 o costos totales de viaje con signos negativos.

## Analisis de Correlación entre variables númericas

# Correlación de Pearson

In [31]:
def correlacion_df(df, target_var, feature_cols, method):
    # Ensamblador de caracteristicas en vectores
    target_var  = [target_var]
    feature_cols  =  feature_cols
    df_cor  =  df.select(target_var+feature_cols)
    assembler =  VectorAssembler(inputCols  =  target_var+feature_cols, outputCol =  "features")
    df_cor =  assembler.transform(df_cor)

    # Calculo de matriz de correlación
    correlation_matrix =  Correlation.corr(df_cor, "features", method  =  method).head()

    # Extracción de coeficientes de correlación
    target_corr_list = [correlation_matrix[0][i, 0] for i in range(len(feature_cols)+1)][1:]

    # Creación de dataframe
    correlation_data = [(feature_cols[i], float(target_corr_list[i])) for i in range(len(feature_cols))]

    correlation_df =  spark.createDataFrame(correlation_data, ["variable", "correlación"])
    # Impresión de resultados
    return correlation_df

# Correlación de Pearson
tipos_numericos  =  ["double", "integer"]
columnas_a_excluir = [
    "total_amount",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude"
]
columnas_numericas = [col_name for col_name, col_type in df.dtypes if col_type in tipos_numericos and col_name not in columnas_a_excluir]

# Variable Objetivo
target = 'total_amount'
matrix_df = correlacion_df(df =  df.sample(0.15),
                           target_var =  target,
                           feature_cols =  columnas_numericas,
                           method  =  'pearson')
matrix_df.show()



+--------------------+--------------------+
|            variable|         correlación|
+--------------------+--------------------+
|       trip_distance|0.002471988774011...|
|         fare_amount|  0.9781757024078549|
|               extra| 0.11368099987954366|
|             mta_tax| -0.2094814437513054|
|          tip_amount|  0.6907591358482428|
|        tolls_amount|  0.5853748923321035|
|improvement_surch...| 0.03937315697714953|
+--------------------+--------------------+



Por otro lado, a modo de ilustración se realiza un análisis de correlación entre la variable total amount y algunas variables númericas utilizando 15% del conjunto de datos. Algo que llama demasiado la atención es el hecho de que la correlación entre la distancia del viaje y el costo total del mismo es casi nula, lo cual va en contra la logica del negocio. Una probable explicación podría ser la presencia de viajes con distancia total igual a 0 o costos totales negativos. Lo más recomendable en este caso, sería realizar una limpieza del conjunto de datos. Por otro lado, se observan correlaciones positivas  con tip_amount, y tolls_amount, aunque las correlaciones son moderadas, además de relaciones negativas con mta_tax e improvement_surcharge.

# Data Profiling utilizando ydata-profiling

In [36]:
df_muestra =  df.sample(withReplacement =  False, fraction  =  0.01).limit(10000).toPandas()

reporte  =  ProfileReport(df_muestra,
                          title  =  "Data Profiling NYC Taxi Data (Spark)",
                          explorative  =  True)
reporte.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 19/19 [00:00<00:00, 40.70it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Utilizando 10,000 registros como muestra, se observa que el conjunto de datos presenta problema de desbalanceo de clases en las variables RatecodeID, fwd_flag o payment_type, por ejemplo. Además de problema de multicolinealidad. Finalmente se observan valores faltantes en 7 columnas, aunque el problema más severo se encuentra en la columna tolls_amount con cerca del 95% de valores faltantes.